# V2 — RAW generation (Kaggle)

Project files: `/kaggle/input/datasets/kamilml/sources/` (upload the full `Steganography_benchmarks_V2/` folder)

**Kaggle does not cache models** — always downloads from Hugging Face (`--platform kaggle`).

Requires: Internet + secret `HF_TOKEN`.

**Important — saving results:** Kaggle wipes `/kaggle/working` after the session closes/resets. To keep data:
1. After generation, download `runs_backup_*.zip` from the **Output** tab (right panel), or
2. **Save Version → Quick Save** with *Save output* enabled, or
3. Set `RESUME_RUN_DIR` and resume interrupted generation (files are appended per task).

In [3]:
import os
import shutil
from pathlib import Path

INPUT_DIR = Path('/kaggle/input/datasets/kamilml/sources1')
PROJECT_DIR = Path('/kaggle/working/Steganography_benchmarks_V2')

SCRIPTS_SRC = INPUT_DIR / 'scripts' if (INPUT_DIR / 'scripts').is_dir() else INPUT_DIR
V2_FILES = [
    'generate_responses.py',
    'evaluate_responses.py',
    'eval_handlers.py',
    'common.py',
    'code_extract.py',
    'prompts.py',
    'model_runtime.py',
    'raw_store.py',
    'dummy_processor.py',
    'humaneval_subset_eval.py',
    'perplexity_metrics.py',
    'binoculars_scorer.py',
    'notebook_runner.py',
    'requirements.txt',
]

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'scripts').mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'runs').mkdir(parents=True, exist_ok=True)

missing = [name for name in V2_FILES if not (SCRIPTS_SRC / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing in {SCRIPTS_SRC}: {missing}')

for name in V2_FILES:
    shutil.copy2(SCRIPTS_SRC / name, PROJECT_DIR / 'scripts' / name)
    print(f'copied scripts/{name}')

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())


copied generate_responses.py
copied evaluate_responses.py
copied eval_handlers.py
copied common.py
copied code_extract.py
copied prompts.py
copied model_runtime.py
copied raw_store.py
copied dummy_processor.py
copied humaneval_subset_eval.py
copied perplexity_metrics.py
copied binoculars_scorer.py
copied notebook_runner.py
copied requirements.txt
cwd: /kaggle/working/Steganography_benchmarks_V2


In [4]:
!pip install -q -r scripts/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.1 MB/s eta 0:00:00


In [5]:
import os
from kaggle_secrets import UserSecretsClient

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [6]:
# --- CONFIG ---
TEST = 'humaneval'        # humaneval | perplexity | capacity | binoculars
MODEL = 'gemma'
THRESHOLD = 0.1          # run the cell 3×: 0.01, 0.05, 0.1
TOP_N = 16
MAX_NEW_TOKENS = 512
HUMANEVAL_TASKS = '74-163'  # zadania 74..163 (91 szt.); None = wszystkie 164
RESUME_RUN_DIR = None    # e.g. 'runs/2026-07-11_...' — resume after interrupt

import shutil
import sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, 'scripts')
from notebook_runner import run_live


def backup_runs() -> None:
    """Zip runs/ → /kaggle/working (Output tab). Also runs after errors."""
    runs = Path('runs')
    if not runs.exists():
        print('No runs/ folder — nothing to back up.')
        return
    jsonl_files = list(runs.rglob('*.jsonl'))
    if not jsonl_files:
        print('No *.jsonl files in runs/.')
        return
    for p in jsonl_files:
        lines = sum(1 for _ in p.open(encoding='utf-8'))
        print(f'  {p.relative_to(runs)}: {lines} linii')
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
    archive_base = out_dir / f'runs_backup_{ts}'
    archive = shutil.make_archive(str(archive_base), 'zip', runs)
    print(f'Backup zapisany: {archive}')
    print('Download from the Output tab (right panel) BEFORE you close the session.')


print('cwd:', Path.cwd())
print('scripts/generate_responses.py:', Path('scripts/generate_responses.py').exists())

if RESUME_RUN_DIR:
    cmd = [sys.executable, 'scripts/generate_responses.py', '--run-dir', str(RESUME_RUN_DIR)]
else:
    cmd = [
        sys.executable,
        'scripts/generate_responses.py',
        '--test', TEST,
        '--model', MODEL,
        '--threshold', str(THRESHOLD),
        '--top-n', str(TOP_N),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--platform', 'kaggle',
        '--output-root', 'runs',
    ]
    if HUMANEVAL_TASKS and TEST == 'humaneval':
        cmd += ['--humaneval-tasks', HUMANEVAL_TASKS]

print('cmd:', ' '.join(cmd))

try:
    run_live(cmd)
except Exception as exc:
    print(f'\nGeneration error: {exc}')
    print('Partial data may be in runs/ — creating backup...')
finally:
    backup_runs()

print('\nAfter downloading the zip: copy into results/<benchmark>/runs/ and run python results/run_evaluate.py <run_name>')


usage: generate_responses.py [-h] [--model MODEL] [--threshold THRESHOLD]
                             [--top-n TOP_N] [--max-new-tokens MAX_NEW_TOKENS]
                             [--seed SEED] [--humaneval-tasks HUMANEVAL_TASKS]
                             [--output-root OUTPUT_ROOT] [--run-dir RUN_DIR]
                             [--platform {colab,kaggle}]
                             [--model-cache-dir MODEL_CACHE_DIR] [--offline]
                             [--list-models]
generate_responses.py: error: unrecognized arguments: --test humaneval

Generation error: Command '['python', '-u', 'generate_responses.py', '--test', 'humaneval', '--model', 'gemma', '--threshold', '0.1', '--top-n', '16', '--max-new-tokens', '512', '--platform', 'kaggle', '--output-root', 'runs', '--humaneval-tasks', '74-163']' returned non-zero exit status 2.
Partial data may be in runs/ — creating backup...
No runs/ folder — nothing to back up.

After downloading the zip, evaluate locally: python results

In [7]:
# Run manually if the session is still alive (e.g. after an error in another cell)
from pathlib import Path

runs = Path('runs')
if runs.exists():
    for p in sorted(runs.rglob('*.jsonl')):
        n = sum(1 for _ in p.open(encoding='utf-8'))
        print(f'{p}: {n} samples')
else:
    print('No runs/ — the session may have been reset; data is gone.')

# backup_runs()  # uncomment if the generation cell did not reach finally

No runs/ — the session may have been reset; data is gone.
